# Begleitnotizbuch zu Kapitel 6
**Build Your First LLM — Kapitel 6: NumPy & PyTorch Überlebensführer**

Führe diese Zellen von oben nach unten aus, um die Tensor-Grundlagen, Maskierung, Broadcasting und eine kleine Trainingsschleife zu sehen.

**Route für den ersten Durchgang:** Setup → Zahlenbeispiele (Formen, Achsen, Matrixprodukte) → Vorschau auf Attention und Einbettungen → Kontrollpunkt und Lernkurve. Die Vorschauen kannst du in Kapitel 9 und 10 vertiefen.

In [ ]:
# ===== IMPORTE =====
# torch: Die PyTorch-Bibliothek für Tensor-Operationen (denke: mehrdimensionale Arrays)
# numpy: Die klassische Bibliothek für numerisches Rechnen (PyTorch ist davon inspiriert)
# torch.nn: Bausteine für neuronale Netze (Schichten, etc.)
import torch, numpy as np
import torch.nn as nn

print('Torch-Version:', torch.__version__)
print('CUDA verfügbar:', torch.cuda.is_available())
# CUDA = GPU-Computing. Wenn True, können wir GPU-Beschleunigung nutzen (10-100× schneller für Deep Learning)

## Was ist ein Tensor?

Ein **Tensor** ist ein mehrdimensionales Array von Zahlen:
- **1D-Tensor** = eine Liste `[1, 2, 3]` (wie eine Zeile in einer Tabelle)
- **2D-Tensor** = eine Tabelle/Matrix (Zeilen und Spalten)
- **3D-Tensor** = ein Stapel von Tabellen (wie mehrere Blätter in Excel)

**Warum PyTorch statt NumPy?**
Beide verarbeiten mehrdimensionale Arrays, aber PyTorch fügt hinzu:
1. **GPU-Unterstützung** — Daten auf GPU verschieben für 10-100× Geschwindigkeitsgewinn
2. **Automatische Gradienten** — berechnet Ableitungen für das Training neuronaler Netze
3. **Neuronale Netz-Schichten** — vorgefertigte Bausteine

## Tensoren erstellen
Aus Python/NumPy-Daten und mit gängigen Füllregeln.

In [ ]:
# ===== Tensoren erstellen =====

# Aus Python-Daten
data = torch.tensor([[1, 2, 3], [4, 5, 6]])  # 2×3 Tensor

# Füllregeln (erstelle Tensoren mit spezifischen Werten gefüllt)
zeros = torch.zeros(3, 4)     # 3×4 Tensor mit Nullen
ones = torch.ones(2, 3, 4)    # 2×3×4 Tensor mit Einsen
uniform = torch.rand(3, 4)    # Zufallswerte im Bereich [0, 1)
normal = torch.randn(3, 4)    # Zufallswerte aus Normalverteilung (Mittelwert=0, Standardabweichung=1)
integers = torch.randint(0, 10, (3, 4))  # Zufällige Ganzzahlen im Bereich [0, 10)

# Bereichssequenzen (wie Pythons range)
sequence = torch.arange(0, 10, 2)  # [0, 2, 4, 6, 8]
linspace = torch.linspace(0, 1, 5)  # 5 gleichmässig verteilte Punkte von 0 bis 1

# Datentypen (dtype) — Präzision kontrollieren
float_tensor = torch.tensor([1.0, 2.0], dtype=torch.float32)  # 32-Bit Fliesskommazahlen (Standard)
int_tensor = torch.tensor([1, 2], dtype=torch.long)           # 64-Bit Ganzzahlen (für Indizes)

# NumPy ↔ PyTorch (sie teilen sich den Speicher — Änderungen an einem betreffen das andere!)
np_data = np.array([1, 2, 3], dtype=np.float32)
torch_from_np = torch.from_numpy(np_data)  # teilt sich Speicher mit np_data
back_to_np = torch_from_np.numpy()         # zurück zu NumPy

# Geräteplatzierung — CPU oder GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_tensor = torch.randn(3, 4, device=device, dtype=torch.float32)

print('zeros Form:', zeros.shape)
print('gpu_tensor Gerät:', gpu_tensor.device)

## Reproduzierbarkeit: Seeds setzen
Kritisch für Debugging und Vergleich von Experimenten

In [ ]:
# Zufallszustand für Reproduzierbarkeit fixieren
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Diese sind jetzt bei jedem Durchlauf identisch
a = torch.randn(3, 4)
b = torch.randn(3, 4)

# Ohne Seed — Ergebnisse ändern sich bei jedem Durchlauf
# Aber mit Seed sind sie reproduzierbar
torch.manual_seed(42)
x1 = torch.randn(2, 3)
torch.manual_seed(42)
x2 = torch.randn(2, 3)
print('Mit Seed stimmen Tensoren überein:', torch.allclose(x1, x2))

# Warum das wichtig ist: Reproduzierbare Experimente für Debugging und Forschung

## Von Listen zu Tensoren: Dimensionen aufbauen
Verbinde die einfachen Listen aus Kapitel 5 mit mehrdimensionalen Tensoren

In [ ]:
# Schritt 1: 1D-Tensoren (Kapitel 5 Rückblick)
# Aus Kapitel 5: Token-IDs vom Tokenizer
token_ids = [2, 3, 4, 6]
tokens_1d = torch.tensor(token_ids)
print(f"1D Form: {tokens_1d.shape}")  # torch.Size([4])
print(f"Daten: {tokens_1d}")

In [ ]:
# Schritt 2: Batching (1D → 2D)
# Mehrere Sätze gleichzeitig verarbeiten
batch = torch.tensor([
    [2, 3, 4, 6],    # Satz 1
    [5, 7, 8, 9]     # Satz 2
])
print(f"2D-Batch Form: {batch.shape}")  # torch.Size([2, 4])
print("Erster Satz:", batch[0])
print("Zweites Token des ersten Satzes:", batch[0, 1])

In [ ]:
# Schritt 3: Einbettungen (2D → 3D)
# 768 Zahlen pro Token hinzufügen (GPT-2 Stil)
embeddings_3d = torch.randn(2, 4, 768)
print(f"3D-Einbettungen Form: {embeddings_3d.shape}")  # torch.Size([2, 4, 768])

# Durch Dimensionen navigieren: Batch → Token → Merkmale
print("Erste Satz-Einbettungen:", embeddings_3d[0].shape)      # (4, 768)
print("Erstes Token des ersten Satzes:", embeddings_3d[0, 0].shape)  # (768,)

In [ ]:
# Schritt 4: Aufmerksamkeitsköpfe Vorschau (3D → 4D)
# Multi-Head-Attention fügt eine weitere Dimension hinzu (keine Sorge um Details)
attention_4d = torch.randn(2, 8, 4, 4)  # (Batch, Köpfe, Sequenz, Sequenz)
print(f"4D-Attention Form: {attention_4d.shape}")
print("Form-Interpretation: (Batch, Köpfe, Sequenzlänge, Sequenzlänge)")

## Umformen, Quetschen, Permutieren
Form-Gymnastik, die du ständig verwenden wirst.

In [ ]:
# ===== Umformoperationen =====
# Umformen = Daten reorganisieren ohne Werte zu ändern (wie Kisten in einem Lager umordnen)

x = torch.arange(12)               # 1D-Tensor erstellen [0,1,2,...,11], Form (12,)
print(f'Start: {x.shape}')

# view() — umformen, erfordert aber "zusammenhängenden" Speicher (Daten sequenziell angeordnet)
x = x.view(3, 4)                   # Umformen zu (3, 4): 3 Zeilen × 4 Spalten
print(f'Nach view(3,4): {x.shape}')

# reshape() — wie view(), behandelt aber nicht-zusammenhängende Tensoren (sicherer, nutze dies bei Unsicherheit)
x = x.reshape(2, 2, 3)             # Umformen zu (2, 2, 3)
print(f'Nach reshape(2,2,3): {x.shape}')

# -1 bedeutet "finde diese Dimension für mich heraus"
x = x.view(-1, 3)                  # -1 wird zu 4 (12 Gesamtelemente ÷ 3 = 4)
print(f'Nach view(-1,3): {x.shape}')

# unsqueeze/squeeze — Dimensionen der Grösse 1 hinzufügen oder entfernen
y = torch.randn(3, 4)
y = y.unsqueeze(0)                 # (1, 3, 4) — Batch-Dimension an Position 0 hinzufügen
y = y.unsqueeze(-1)                # (1, 3, 4, 1) — Dimension am Ende hinzufügen
y = y.squeeze()                    # ALLE Dimensionen der Grösse 1 entfernen → (3, 4)
print(f'Nach squeeze: {y.shape}')

# permute — Dimensionen neu anordnen (wie Transponieren, aber für beliebige Anzahl von Dimensionen)
z = torch.randn(2, 3, 4)           # (Batch, Sequenz, Merkmale)
z = z.permute(0, 2, 1)             # (Batch, Merkmale, Sequenz) — letzten zwei Dimensionen tauschen
print(f'Nach permute: {z.shape}')

# flatten — Dimensionen zusammenklappen
t = torch.randn(2, 3, 4)
t_flat = t.flatten()               # (24,) — alle Dimensionen zusammengeklappt
t_flat_features = t.flatten(1)     # (2, 12) — abflachen ab Dimension 1
print(f'Vollständig abgeflacht: {t_flat.shape}')
print(f'Merkmale abflachen: {t_flat_features.shape}')

# Bei Matrizen tauscht .T die beiden Achsen; im Batch nur die letzten beiden tauschen.
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])
print(a @ b)
print(a.T)
x = torch.randn(2, 8, 5, 64)
print((x @ x.transpose(-2, -1)).shape)

## Indizierung und Slicing
Batches, Tokens auswählen und boolesche Masken verwenden.

In [ ]:
# ===== Indizierung und Slicing =====
# Mehrdimensionale Daten navigieren wie du Ordner navigieren würdest: Batch → Token → Merkmale

# Einen 4D-Tensor erstellen, der Attention simuliert: (Batch, Köpfe, Sequenz, Sequenz)
x = torch.randn(2, 4, 6, 6)  # 2 Batches, 4 Köpfe, 6 Tokens, 6 Tokens

# Grundlegende Indizierung
first_batch = x[0]              # Form: (4, 6, 6) - erster Batch, alle Köpfe
first_head = x[0, 0]            # Form: (6, 6) - erster Batch, erster Kopf
single_value = x[0, 0, 0, 0]    # Form: () - eine einzelne Zahl

# Slicing mit Doppelpunkten
first_two_batches = x[:2]       # Form: (2, 4, 6, 6) - erste 2 Batches
all_but_last_token = x[:, :, :-1, :]  # Form: (2, 4, 5, 6) - letztes Token entfernen
every_other_head = x[:, ::2]    # Form: (2, 2, 6, 6) - Köpfe 0 und 2

# Negative Indizes zählen vom Ende
last_token = x[:, :, -1, :]     # Form: (2, 4, 6) - letztes Token in jeder Sequenz

# Boolesche Maskierung (nach Bedingung filtern)
mask = torch.tensor([True, False, True, False])
filtered_heads = x[0, mask]     # Form: (2, 6, 6) - nur Köpfe 0 und 2

print(f'Originalform: {x.shape}')
print(f'Erste Batch-Form: {first_batch.shape}')
print(f'Alle ausser letztes Token: {all_but_last_token.shape}')
print(f'Gefilterte Köpfe: {filtered_heads.shape}')

## Attention: Das Herz der Transformer

**Das grosse Bild:** Attention berechnet einen gewichteten Durchschnitt aller Wörter, wobei die Gewichte aus Relevanz-Scores stammen. Wie beim Lesen von "Die Katze sass auf der Matte" — bei der Verarbeitung von "sass" schaust du zurück zu "Katze" (wer sass?) und "Matte" (wo sass?).

Verfolge die Formen von den Scores bis zur gewichteten Summe. Die kausale Maske blendet spätere Positionen aus.

In [ ]:
import torch
torch.manual_seed(42)

batch, seq_len, d_head = 2, 5, 64
# Q: Wonach suche ich? K: Was biete ich an? V: Welchen Inhalt gebe ich weiter?
Q = torch.randn(batch, seq_len, d_head)
K = torch.randn(batch, seq_len, d_head)
V = torch.randn(batch, seq_len, d_head)

scores = Q @ K.transpose(-2, -1)
print(f"scores: {tuple(scores.shape)}")

scores = scores / (d_head ** 0.5)

causal_mask = torch.tril(torch.ones(seq_len, seq_len))
scores = scores.masked_fill(causal_mask == 0, float('-inf'))

attn_weights = torch.softmax(scores, dim=-1)

output = attn_weights @ V
print(f"output: {tuple(output.shape)}")

print(attn_weights[0, 2])

## Token- und Positionseinbettungen

Jedes LLM beginnt damit, Token-IDs in dichte Vektoren umzuwandeln.

**Das Problem:** Neuronale Netze können keinen rohen Text wie "Katze" verarbeiten. Token-IDs (wie `5` für "Katze") sind willkürlich — ID 5 ist nicht "näher" an 6 als an 500.

**Die Lösung:** Jede Token-ID auf einen gelernten Vektor abbilden (768 Zahlen für GPT-2). Ähnliche Wörter lernen durch Training ähnliche Vektoren.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(42)

vocab_size = 50257
d_model = 768

token_embedding = nn.Embedding(vocab_size, d_model)

token_ids = torch.tensor([[464, 3797, 3332, 319, 262],
                          [464, 2603, 3332, 319, 262]])
print(f"Form der IDs: {token_ids.shape}")

token_vectors = token_embedding(token_ids)
print(f"Form der Vektoren: {token_vectors.shape}")

print(token_embedding.weight.numel())

## Broadcasting-Beispiele
Bias-Addition und Maskierung-Broadcast.

In [ ]:
x = torch.randn(3, 4)
y = x + 5
batch = torch.randn(32, 10, 768)
bias = torch.randn(768)
result = batch + bias  # Bias wird gebroadcastet
scores = torch.randn(4, 8, 10, 10)
mask = torch.triu(torch.ones(1, 1, 10, 10), 1)
masked = scores + mask * -1e9
print('Ergebnisform:', result.shape)

## Wesentliche Operationen: Elementweise, Matmul, Reduktionen

**Elementweise Operationen** arbeiten positionsweise — wie das Addieren von zwei Tabellen Zelle für Zelle. Wenn `a` und `b` beide 3×4 Tensoren sind, dann addiert `a + b` `a[0,0]` zu `b[0,0]`, `a[0,1]` zu `b[0,1]`, und so weiter. Gleiche Form rein, gleiche Form raus.

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(3, 4)
add = a + b
mul = a * b
square = a ** 2
exp = torch.exp(a)
x = torch.randn(32, 10, 64)
W = torch.randn(64, 128)
y = x @ W
total = a.sum()
row_sums = a.sum(dim=1, keepdim=True)
col_means = a.mean(dim=0)
max_vals, max_idx = a.max(dim=1)
c = torch.cat([a, b], dim=0)
d = torch.stack([a, b], dim=0)
logits = torch.randn(3, 5)
probs = torch.softmax(logits, dim=-1)
print('y Form:', y.shape)
print('Probs Zeilensummen:', probs.sum(dim=-1))

## Autograd: Automatische Gradienten

**Was ist ein Gradient?** Die Ableitung — um wie viel ändert sich die Ausgabe, wenn sich die Eingabe ändert?

**Warum wichtig?** Das Training eines neuronalen Netzes bedeutet, Millionen von Parametern anzupassen. Gradienten sagen uns, in welche Richtung wir anpassen sollen. PyTorchs Autograd macht dies automatisch.

In [ ]:
# Einfacher Berechnungsgraph: x → y → z
x = torch.tensor([2.0, 3.0], requires_grad=True)  # Operationen auf x verfolgen
y = x ** 2        # y = [4.0, 9.0]
z = y.sum()       # z = 13.0

# Gradienten automatisch berechnen
z.backward()      # "wie ändert sich z, wenn ich x ändere?"
print('Gradienten:', x.grad)  # tensor([4., 6.])

# Was ist gerade passiert?
# z = (x²).sum() → dz/dx = 2x
# Bei x=[2, 3] sind Gradienten 2*[2, 3] = [4, 6]
# backward() hat dies berechnet, indem es den Graphen rückwärts durchlaufen hat!

print('\n✅ Manuelle Überprüfung: dz/dx = 2x')
print(f'   Bei x=[2, 3]: 2*x = {2 * x.detach()}')

# Wann Verfolgung stoppen (spart Speicher während der Inferenz):
print('\n--- Gradientenverfolgung stoppen ---')

# Option 1: Kontextmanager (für einen Codeblock)
with torch.no_grad():
    y_no_grad = x * 2  # keine Gradientenverfolgung
    print(f'Kein Gradient berechnet: {y_no_grad}')

# Option 2: Detach (für einen einzelnen Tensor)
detached = x.detach()  # neuer Tensor, keine Gradient-Historie
print(f'Abgelöster Tensor: {detached}')

## Kontrollpunkt: Ein Optimierungsschritt

Vergleiche den Verlust auf demselben Batch vor und nach dem Update. Die Reihenfolge lautet: Gradienten löschen, Vorwärtsdurchlauf, Verlust berechnen, rückwärts ableiten, Gewichte aktualisieren.

In [ ]:
import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()  # Erforderlich: Elternklasse initialisieren
        # Schichten definieren (diese haben lernbare Parameter)
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.act = nn.ReLU()

    def forward(self, x):
        # Berechnung definieren: wie x durch die Schichten fliesst
        x = self.fc1(x)      # Lineare Transformation
        x = self.act(x)      # Nichtlinearität (ReLU)
        x = self.norm(x)     # Normalisierung
        x = self.fc2(x)      # Ausgabe-Projektion
        return x

import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)

model = SimpleMLP(16, 32, 2)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
inputs = torch.randn(64, 16)
labels = torch.randint(0, 2, (64,))

optimizer.zero_grad(set_to_none=True)
logits = model(inputs)
loss = loss_fn(logits, labels)
print(f"Vorher: Verlust={loss.item():.4f}")
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

with torch.no_grad():
    after = loss_fn(model(inputs), labels)
print(f"Danach: Verlust={after.item():.4f}")
with torch.no_grad():
    accuracy = (model(inputs).argmax(dim=-1) == labels).float().mean()
print(f"Genauigkeit auf dem Trainingsbatch: {accuracy:.1%} (kein Test der Generalisierung)")

## Beobachte, wie der Verlust sinkt

Kann eine lineare Schicht aus 32 Wertepaaren die Regel `y = 2x + 1` lernen?

In [ ]:
import torch
torch.manual_seed(42)
x = torch.linspace(0, 1, 32).unsqueeze(1)  # 32 Eingaben, Form (32, 1)
y = 2 * x + 1                              # die Regel, die das Modell lernen soll
model = torch.nn.Linear(1, 1)              # ein Gewicht, ein Bias
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = torch.nn.MSELoss()
for step in range(1, 51):
    opt.zero_grad()                        # 1. alte Gradienten löschen
    loss = loss_fn(model(x), y)            # 2+3. Vorwärtsdurchlauf, Fehler messen
    loss.backward()                        # 4. Gradienten berechnen
    opt.step()                             # 5. Gewichte aktualisieren
    if step % 10 == 0:
        print(f"Schritt {step:2d}: Verlust = {loss.item():.6f}")
print(f"gelernt: y = {model.weight.item():.2f}x + {model.bias.item():.2f}")

## Zusammenfassung

Du hast jetzt alle wichtigen PyTorch-Operationen für die LLM-Entwicklung gesehen:

✅ **Tensor-Grundlagen** - Erstellung, Datentypen, Geräte
✅ **Reproduzierbarkeit** - Seeds für Debugging
✅ **Dimensionsaufbau** - 1D → 2D → 3D → 4D Progression
✅ **Umformen & Indizierung** - Navigation durch mehrdimensionale Daten
✅ **Aufmerksamkeitsmechanismus** - Scores, Maske und gewichtete Summe
✅ **Kausale Maskierung** - zukünftiges Token-Spicken verhindern
✅ **Einbettungen** - Token-IDs und gelernte Vektoren
✅ **Broadcasting** - automatische Formerweiterung
✅ **Mathematische Operationen** - elementweise, Matmul, Reduktionen
✅ **Autograd** - automatische Differentiation
✅ **Trainingsschleifen** - vorwärts, rückwärts, optimieren mit Gradientenclipping

**Nächste Schritte:** Kapitel 7 zeigt dir, wie du echte Textdaten für das Training vorbereitest!